# Modul 8: Fehler, Debugging & Ethik — Wenn Agents Mist bauen

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook simulierst du 5 typische Agent-Fehlermodi und baust einen Debugging-Flowchart.

🎯 **Lernziel:** Die 5 Fehlermodi erkennen, systematisch debuggen und Ethik-Regeln definieren.

---

## Die 5 Agent-Fehlermodi

```
  1. HALLUZINATION     → Agent erfindet Fakten
  2. TOOL-MISUSE       → Falsches Tool, falsche Parameter
  3. LOOP/RECURSION    → Agent dreht sich im Kreis
  4. CONTEXT-OVERFLOW  → Kontext zu lang, Info geht verloren
  5. UNBEABSICHTIGT    → Richtig ausgeführt, aber falsche Aktion
```

**Debugging-Prinzip:** Logs → Hypothese → Isolieren → Fix

In [ ]:
# Simulation aller 5 Fehlermodi

import random

class AgentSimulator:
    """Simuliert einen Agent der verschiedene Fehler produziert."""

    def __init__(self):
        self.log = []
        self.tool_calls = 0
        self.max_iterations = 10

    def _log(self, level, msg):
        entry = f'[{level}] {msg}'
        self.log.append(entry)
        print(f'  {entry}')

    def simulate_hallucination(self):
        """Fehlermodus 1: Agent erfindet Fakten."""
        print('\n--- Fehlermodus 1: HALLUZINATION ---')
        fake_facts = [
            'Python wurde 2015 von Elon Musk erfunden.',
            'Die Erde hat 3 Monde.',
            'OpenClaw wurde 1999 released.',
        ]
        self._log('THINK', 'User fragt nach Python-Geschichte...')
        fact = random.choice(fake_facts)
        self._log('RESPOND', f'Antwort: {fact}')
        self._log('ERROR', 'HALLUZINATION erkannt: Fakt ist falsch!')
        return {'type': 'hallucination', 'output': fact, 'fix': 'Quellen-Check einbauen, Grounding erzwingen'}

    def simulate_tool_misuse(self):
        """Fehlermodus 2: Falsches Tool aufgerufen."""
        print('\n--- Fehlermodus 2: TOOL-MISUSE ---')
        self._log('THINK', 'User will Datei lesen...')
        self._log('ACT', 'Tool: file_write("wichtig.txt", "leer")')  # FALSCH!
        self._log('ERROR', 'TOOL-MISUSE: file_write statt file_read!')
        return {'type': 'tool_misuse', 'output': 'Datei ueberschrieben statt gelesen', 'fix': 'Tool-Beschreibungen praeizisieren, Read-only Policy'}

    def simulate_loop(self):
        """Fehlermodus 3: Endlosschleife."""
        print('\n--- Fehlermodus 3: LOOP/RECURSION ---')
        for i in range(5):
            self._log('THINK', f'Iteration {i+1}: Ergebnis nicht gut genug, nochmal...')
            self._log('ACT', f'web_search("gleiche Frage")')
            self._log('OBSERVE', 'Gleiches Ergebnis wie vorher.')
        self._log('ERROR', f'LOOP erkannt: {5} identische Iterationen!')
        return {'type': 'loop', 'output': '5 Wiederholungen ohne Fortschritt', 'fix': 'max_iterations setzen, Loop-Detection einbauen'}

    def simulate_context_overflow(self):
        """Fehlermodus 4: Kontext wird zu lang."""
        print('\n--- Fehlermodus 4: CONTEXT-OVERFLOW ---')
        context_size = 0
        max_context = 8000  # Tokens
        for i in range(20):
            tokens = random.randint(200, 800)
            context_size += tokens
            if context_size > max_context:
                self._log('ERROR', f'OVERFLOW: Kontext {context_size} > {max_context} Tokens!')
                self._log('WARN', 'Aelteste Nachrichten gehen verloren...')
                break
            self._log('INFO', f'Turn {i+1}: +{tokens} Tokens (gesamt: {context_size})')
        return {'type': 'context_overflow', 'output': f'Kontext bei {context_size} Tokens', 'fix': 'Kontext-Fenster managen, Zusammenfassungen nutzen'}

    def simulate_unintended_action(self):
        """Fehlermodus 5: Technisch korrekt, aber falsche Aktion."""
        print('\n--- Fehlermodus 5: UNBEABSICHTIGTE AKTION ---')
        self._log('THINK', 'User sagt "Loesch die alten Dateien"...')
        self._log('ACT', 'exec("rm -rf /important/data/")')  # Richtig ausgefuehrt, aber FALSCH verstanden
        self._log('OBSERVE', 'Dateien geloescht.')
        self._log('ERROR', 'UNBEABSICHTIGT: User meinte Log-Dateien, nicht Projektdaten!')
        return {'type': 'unintended', 'output': 'Falsche Dateien geloescht', 'fix': 'Approval-Policy fuer destruktive Aktionen, Rueckfrage bei Ambiguitaet'}


# Alle 5 Fehlermodi durchspielen
sim = AgentSimulator()
errors = [
    sim.simulate_hallucination(),
    sim.simulate_tool_misuse(),
    sim.simulate_loop(),
    sim.simulate_context_overflow(),
    sim.simulate_unintended_action(),
]

print('\n\n=== Zusammenfassung der Fehlermodi ===')
print(f'{"Typ":<22} {"Fix"}')
print('-' * 70)
for e in errors:
    print(f'{e["type"]:<22} {e["fix"]}')

In [ ]:
# Debugging-Flowchart als Code

def debug_agent_error(log_entries):
    """Systematischer Debugging-Flowchart fuer Agent-Fehler."""
    print('=== Debugging-Flowchart ===\n')

    # Schritt 1: Fehler-Typ identifizieren
    error_entries = [e for e in log_entries if '[ERROR]' in e]
    if not error_entries:
        print('  Kein Fehler in Logs gefunden.')
        return 'no_error'

    print(f'  1. FEHLER GEFUNDEN: {len(error_entries)} Error(s)')
    for e in error_entries:
        print(f'     {e}')

    # Schritt 2: Kategorie bestimmen
    full_log = ' '.join(log_entries).lower()

    if 'halluzination' in full_log or 'falsch' in full_log:
        category = 'hallucination'
        print('  2. KATEGORIE: Halluzination')
        print('  3. FIX: Grounding + Quellencheck erzwingen')
    elif 'tool-misuse' in full_log or 'falsches tool' in full_log:
        category = 'tool_misuse'
        print('  2. KATEGORIE: Tool-Misuse')
        print('  3. FIX: Tool-Beschreibungen verbessern + Policy verschaerfen')
    elif 'loop' in full_log or 'iteration' in full_log:
        category = 'loop'
        print('  2. KATEGORIE: Loop/Recursion')
        print('  3. FIX: max_iterations + Loop-Detection')
    elif 'overflow' in full_log:
        category = 'context_overflow'
        print('  2. KATEGORIE: Context-Overflow')
        print('  3. FIX: Kontext-Management + Summarization')
    elif 'unbeabsichtigt' in full_log:
        category = 'unintended'
        print('  2. KATEGORIE: Unbeabsichtigte Aktion')
        print('  3. FIX: Approval-Policy + Rueckfrage bei Ambiguitaet')
    else:
        category = 'unknown'
        print('  2. KATEGORIE: Unbekannt')
        print('  3. FIX: Logs genauer analysieren')

    # Schritt 3: Schweregrad
    severity = 'HOCH' if category in ['unintended', 'tool_misuse'] else 'MITTEL'
    print(f'  4. SCHWEREGRAD: {severity}')

    return category

# Test mit den gesammelten Logs
result = debug_agent_error(sim.log)

In [ ]:
# 🎯 ÜBUNG: Baue einen Loop-Detektor
#
# Aufgabe: Schreibe eine Funktion die erkennt ob ein Agent
# in einer Schleife steckt (gleiche Tool-Aufrufe wiederholt).
# Return: True wenn Loop erkannt, False wenn OK.

def detect_loop(tool_calls, threshold=3):
    """Erkennt ob die letzten N Tool-Aufrufe identisch sind."""
    # TODO: Prüfe ob die letzten 'threshold' Aufrufe gleich sind
    # Tipp: Vergleiche tool_calls[-threshold:]
    pass

# Test-Daten
calls_ok = ['web_search("python")', 'file_read("data.txt")', 'summarize("text")']
calls_loop = ['web_search("agent")', 'web_search("agent")', 'web_search("agent")', 'web_search("agent")']

# print(detect_loop(calls_ok))    # Sollte False sein
# print(detect_loop(calls_loop))  # Sollte True sein

In [ ]:
# ✅ LÖSUNG

def detect_loop(tool_calls, threshold=3):
    """Erkennt ob die letzten N Tool-Aufrufe identisch sind."""
    if len(tool_calls) < threshold:
        return False
    last_n = tool_calls[-threshold:]
    return len(set(last_n)) == 1  # Alle gleich = Loop

# Tests
calls_ok = ['web_search("python")', 'file_read("data.txt")', 'summarize("text")']
calls_loop = ['web_search("agent")', 'web_search("agent")', 'web_search("agent")', 'web_search("agent")']
calls_partial = ['web_search("a")', 'web_search("b")', 'web_search("b")', 'web_search("b")']

print(f'OK-Sequenz:      Loop={detect_loop(calls_ok)}')       # False
print(f'Loop-Sequenz:    Loop={detect_loop(calls_loop)}')     # True
print(f'Partial-Loop:    Loop={detect_loop(calls_partial)}')  # True (letzte 3 gleich)
print(f'Kurze Sequenz:   Loop={detect_loop(["a"])}')          # False (zu kurz)

## Ethik-Checkliste

Wann soll ein Agent **NICHT** handeln?

| Situation | Aktion | Warum |
|-----------|--------|-------|
| Medizinische Frage | Nicht diagnostizieren | Haftung, Risiko |
| Persönliche Daten | Nicht speichern ohne Consent | Datenschutz |
| Destruktive Befehle | Rückfrage stellen | Unumkehrbar |
| Unsichere Quellen | Nicht als Fakt darstellen | Fehlinformation |
| Bias erkannt | Transparent machen | Fairness |

---

### ✅ Self-Check
- [ ] Ich kann die 5 Fehlermodi benennen und je ein Beispiel geben
- [ ] Ich kann den Debugging-Flowchart anwenden (Logs → Hypothese → Fix)
- [ ] Ich habe eine Ethik-Checkliste für meinen Agent

**→ Weiter zu [Modul 9: Multi-Agent Systeme](modul9_multi_agent.ipynb)**